[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [LightGBM and XGBoost, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)

# Categorical Features &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import mean_squared_error

TERRAINS = ["coastal", "fjord", "inland", "valley", "mountain"]


def station_days(n_stations=400, days=50, seed=2026):
    """Made-up daily minimum temperatures at weather stations in Norway, one row per station and day."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n_stations)
    coast_km = np.minimum(rng.exponential(30, n_stations), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n_stations) + 2.5 * coast_km, 0, 1600)
    terrain = rng.choice(TERRAINS, n_stations, p=[0.2, 0.2, 0.25, 0.2, 0.15])
    has_snow_sensor = rng.uniform(size=n_stations) < 0.7
    offset = rng.normal(0, 0.8, n_stations)
    station = np.repeat(np.arange(n_stations), days)
    n = n_stations * days
    day = rng.integers(1, 366, n)
    cloud = np.clip(rng.beta(1.2, 1.0, n), 0, 1)
    wind = rng.gamma(2.0, 2.0, n)
    season = np.cos(2 * np.pi * (day - 20) / 365)
    lat, elev, coast = latitude[station], elevation[station], coast_km[station]
    snow = np.where(season > 0.2, np.clip(rng.normal(30 * season + 0.03 * elev + 2 * (lat - 58), 15), 0, None), 0.0)
    shelter = pd.Series(terrain[station]).map({"coastal": 0.4, "fjord": 1.0, "inland": 1.2, "valley": 1.7, "mountain": 0.8}).to_numpy()
    mean = 8 - 0.55 * (lat - 58) - 0.006 * elev - 9 * season + 3.5 * np.exp(-coast / 40) * season
    cooling = 7 * (1 - cloud) * np.exp(-wind / 3) * (0.4 + (season + 1) / 2) * shelter
    tmin = mean - 2.5 - cooling - 2.0 * (snow > 10) * (1 - cloud) + offset[station] + rng.normal(0, 1.2, n)
    snow_cm = np.where(has_snow_sensor[station], snow.round(0), np.nan)
    return pd.DataFrame({
        "station": [f"ST{s + 1:04d}" for s in station],
        "latitude": lat.round(2), "elevation_m": elev.round(0).astype(int), "coast_km": coast.round(1),
        "terrain": pd.Categorical(terrain[station], categories=TERRAINS),
        "day_of_year": day, "cloud_cover": cloud.round(2), "wind_ms": wind.round(1), "snow_cm": snow_cm,
        "tmin": tmin.round(1),
    })


FEATURES = ["latitude", "elevation_m", "coast_km", "terrain", "day_of_year", "cloud_cover", "wind_ms", "snow_cm"]
days = station_days()
stations = days["station"].unique()
order = np.random.default_rng(0).permutation(len(stations))
train = days[days["station"].isin(stations[order[:300]])]
valid = days[days["station"].isin(stations[order[300:350]])]
test = days[days["station"].isin(stations[order[350:]])]

print("lightgbm", lgb.__version__, "| xgboost", xgb.__version__, "|", len(train), "training rows,",
      len(valid), "validation rows,", len(test), "test rows")

lightgbm 4.6.0 | xgboost 3.4.1 | 15000 training rows, 2500 validation rows, 2500 test rows


The worked examples' models and helpers, rebuilt here from the same made-up stations so that each answer
runs on its own.

In [2]:
y_train, y_valid, y_test = train["tmin"], valid["tmin"], test["tmin"]
lightgbm_native = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.1, random_state=0, verbose=-1)
lightgbm_native.fit(train[FEATURES], y_train)
xgboost_native = xgb.XGBRegressor(n_estimators=300, learning_rate=0.1, random_state=0)
xgboost_native.fit(train[FEATURES], y_train)


def station_day(terrains, categories):
    rows = pd.concat([test[FEATURES].iloc[[0]]] * len(terrains), ignore_index=True)
    rows["terrain"] = pd.Categorical(terrains, categories=categories)
    return rows

**1.** Fit an XGBoost model like `xgboost_native` with `max_cat_to_onehot=6`, under a new name, and print its validation loss and the categories of its first five terrain splits as names.

In [3]:
one_against_rest = xgb.XGBRegressor(n_estimators=300, learning_rate=0.1, random_state=0, max_cat_to_onehot=6)
one_against_rest.fit(train[FEATURES], y_train)
print("validation loss:", round(mean_squared_error(y_valid, one_against_rest.predict(valid[FEATURES])), 3))
splits = one_against_rest.get_booster().trees_to_dataframe()
for codes in splits.loc[splits["Feature"] == "terrain", "Category"].head(5):
    print([TERRAINS[code] for code in codes])

validation loss: 2.553
['coastal']
['coastal']
['coastal']
['coastal']
['coastal']


The first five terrain splits list a single category, coastal, and the validation loss is 2.553, the same as XGBoost's on one-hot columns in the worked examples, since a split on one category is the split a one-hot column allows.

**2.** Fit LightGBM on the one-hot table, the `"one-hot"` entry of `versions`, under a new name, and print how many of its 300 trees split on at least one terrain column, beside the native model's count.

In [4]:
one_hot_train = pd.get_dummies(train[FEATURES], columns=["terrain"], dtype=float)
one_hot = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.1, random_state=0, verbose=-1)
one_hot.fit(one_hot_train, y_train)

one_hot_splits = one_hot.booster_.trees_to_dataframe()
on_columns = one_hot_splits[one_hot_splits["split_feature"].str.startswith("terrain_", na=False)]
native_splits = lightgbm_native.booster_.trees_to_dataframe()
print("one-hot:", on_columns["tree_index"].nunique(), "trees")
print("native:", native_splits.loc[native_splits["split_feature"] == "terrain", "tree_index"].nunique(), "trees")

one-hot: 220 trees
native: 174 trees


220 of the one-hot model's 300 trees split on a terrain column, against 174 for native categories. A split on a one-hot column asks about one terrain, so separating coastal, fjord and mountain stations from the rest takes more than one split.

**3.** Make a table from the first test row with `terrain` written as `"Coastal"`, with a capital, and print what `lightgbm_native` predicts for it and what `xgboost_native` does with it. Then make the column lower case with the categories in `TERRAINS`, and predict again.

In [5]:
capital = station_day(["Coastal"], ["Coastal"])
print("LightGBM:", lightgbm_native.predict(capital).round(3))
try:
    xgboost_native.predict(capital)
except xgb.core.XGBoostError as error:
    print("XGBoostError:", str(error).split(": ", 1)[1].splitlines()[0])

cleaned = capital.assign(terrain=pd.Categorical(capital["terrain"].astype(str).str.lower(), categories=TERRAINS))
print("LightGBM:", lightgbm_native.predict(cleaned).round(3), "| XGBoost:", xgboost_native.predict(cleaned).round(3))

LightGBM: [0.137]
XGBoostError: Found a category not in the training set for the 3th (0-based) column: `Coastal`
LightGBM: [0.528] | XGBoost: [0.457]


LightGBM predicts 0.137 for `Coastal`, its prediction for a missing terrain, and XGBoost raises, since neither model has seen the name with a capital. In lower case with the training categories, both predict what they predict for a coastal station, 0.528 and 0.457.

**4.** In a copy of the test table, give `terrain` its categories in a shuffled order with `.cat.reorder_categories(...)`, and check that LightGBM's test loss is unchanged.

In [6]:
shuffled = [str(name) for name in np.random.default_rng(0).permutation(TERRAINS)]
reordered = test[FEATURES].assign(terrain=test["terrain"].cat.reorder_categories(shuffled))
print(shuffled)
print("original:", round(mean_squared_error(y_test, lightgbm_native.predict(test[FEATURES])), 3))
print("reordered:", round(mean_squared_error(y_test, lightgbm_native.predict(reordered)), 3))

['inland', 'mountain', 'valley', 'coastal', 'fjord']
original: 2.542
reordered: 2.542


The shuffled order gives every terrain a new code, and the test loss stays at 2.542, because LightGBM reads the codes through the table's own list of names.

---

&#8592; **Back to:** [Categorical Features](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/09-categorical-features.ipynb)  &nbsp;&middot;&nbsp;  [LightGBM and XGBoost, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)
